# Scriven Testcase
This is a testcase for evaporation.  
A Vapor Bubble is growing in a superheated liquid.
Initially $t = 0$ there is no vapor in the system.  
The simulation is started from an analytical solution at $t_0 > 0$.  
We can then compare the simulated interface position in time to the analytically predicted.  
See `10.1016/j.ijheatmasstransfer.2021.121233`  

This worksheet also documents the results published in Rieckmann et al. (2024) `10.1016/j.jcp.2023.112716`

Note the following:
* A deviation from the source is the considerably larger surface tension, to deal with capillary timestep restrictions.
* How to take care of boundaries at the outflows?
* Paraview seems to not read time data correcty. Thus time annotation is missing in videos.
* Grid absolute size ($50\mu m$) is too small, then we get crude (or straight up wrong/empty) cut-cell/level-set rules. => rescale lengths to mm.
* How to initialize the level-set?  
    *   Quadratic:  + exact geometric represantation    - zero gradient at (0,0,0) not good for Stokes-Extension
    *   SDF:        + non-vanishing gradients           - inexact geometric representation; not-converging (at least for low res/amr)

In [ ]:
#r "./binaries/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [ ]:
string proj = "ScrivenProblem_v3";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

## Grid and Boundary/Initial Value configuration

Note, that in contrast to the 1-D Testcases, the interface radius is given in terms of $\alpha_A$ :  
* $R_I(t) = 2 \xi \sqrt{\alpha_A t} $

In [ ]:
static class Constants{
    public static double scale = 1000.0; // scale from 1m = 1000mm
    // careful order of declaration matters!!!
    public static double L = 187.5*1e-6 * Math.Pow(scale, 1);
    public static double Xi = 4.063;

    // material parameters
    public static double rho_A = 958.4*Math.Pow(scale, -3);
    public static double rho_B = 0.597*Math.Pow(scale, -3);

    public static double mu_A = 2.8e-4*Math.Pow(scale, -1);
    public static double mu_B = 1.26e-5*Math.Pow(scale, -1);

    public static double Sigma = 0.059 / 50000.0; // very small surface tension so that capillary timestep is bigger...

    public static double dT = 1.25; // rescale temperature to range [0, 1]
    public static double c_A = dT * 4216.0*Math.Pow(scale, 2);
    public static double c_B = dT * 2030.0*Math.Pow(scale, 2);

    public static double k_A = dT * 0.679*Math.Pow(scale, 1);
    public static double k_B = dT * 0.025*Math.Pow(scale, 1);

    public static double hVap  = 2.258e6*Math.Pow(scale, 2);
    public static double T_sat = 0.0; //373.15;
    public static double T_wall = T_sat + 1.0; //+ 1.25;

    public static double alpha_A = k_A / (c_A*rho_A);
    public static double alpha_B = k_B / (c_B*rho_B);
    public static double eps = 1.0-rho_B/rho_A;

    public static double r0 = 50*1e-6*Math.Pow(scale, 1);
    public static double t0 = Math.Pow(0.5 * r0 / Xi, 2) / alpha_A; // start time, to avoid singular massflux at t=0
    public static double tE = Math.Pow(2 * r0 / (2 * Xi),2)/alpha_A - t0; //1.5*1e-3 - t0; // Endtime, until twice the initial radius is reached
    // capillary timestep , for finest res + highest degree, use this, for comparability?!, Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) => 0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [ ]:
Constants.t0

In [ ]:
// numbr of timesteps and dt
// double dt = Constants.dt((int)(8 * Math.Pow(2, 3)), 2); // value for finest resolution, still based on p3 amr2
double dt = Constants.dt((int)(8 * Math.Pow(2, 3)), 2); // value for finest resolution
dt.Display();
Constants.tE / dt

In [ ]:
// estimate dt for cfl, should be larger than above dt
double h = Constants.L / (8 * Math.Pow(2, 2) * Math.Pow(3 + 1, 1)); // last term for degree of level set, is this /(p+1) or  /(p)^2?
double u = Constants.Xi * Constants.alpha_A / Math.Sqrt(Constants.alpha_A * (Constants.t0));
h / u

In [ ]:
// time until twice the initial radius is reached
Math.Pow(2 * Constants.r0 / (2 * Constants.Xi),2)/Constants.alpha_A - Constants.t0

In [ ]:
static class GridFactory{
    public static GridCommons Grid(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] nodes = GenericBlas.Linspace(0, Constants.L, res + 1);
        var grd = Grid3D.Cartesian3DGrid(nodes, nodes, nodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "pressure_outlet_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[0]) < 1e-6) || (Math.Abs(X[1]) < 1e-6) ||  (Math.Abs(X[2]) < 1e-6))
                return 1;
            else
                return 2;
            return et;
        });

        foreach(int n in Enumerable.Range(1, 16)){
            grd.AddPredefinedPartitioning(n+"Proc", X => {                
                double phi = Math.Atan(X[1]/X[0]);
                double dphi = (Math.PI / 2.0) / n;
                double psi = dphi;
                int i = 0;
                while(psi <= Math.PI / 2.0){
                    if(phi <= psi)
                        break;
                    i += 1;
                    psi += dphi;
                }
                return i;
            });
        }

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }
}

In [ ]:
public static class BoundaryAndInitialValueFactory { 

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfacePos(){");
            stw.WriteLine("         return (X,t) => 2 * " + Constants.Xi + " * Math.Sqrt(" + Constants.alpha_A + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfaceVel(){");
            stw.WriteLine("         return (X,t) => " + Constants.Xi + " * " + Constants.alpha_A + " / Math.Sqrt(" + Constants.alpha_A + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){"); // initialize quadratically
            stw.WriteLine("         bool signed = true;");
            stw.WriteLine("         if(signed)");
            stw.WriteLine("             return InterfacePos()(X,t) - Math.Sqrt(X[0]*X[0] + X[1]*X[1] + X[2]*X[2]);");
            stw.WriteLine("         else");
            stw.WriteLine("             return Math.Pow(InterfacePos()(X,t),2) - (X[0]*X[0] + X[1]*X[1] + X[2]*X[2]);");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityA(double[] X, double t){");
            stw.WriteLine("         return " + Constants.eps + " * InterfaceVel()(X,t);");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityXA(double[] X, double t){");
            stw.WriteLine("         double r = Math.Max(X.L2Norm(), BLAS.MachineEps);");
            stw.WriteLine("         double V = X[0]/r * VelocityA(X,0.0);");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return V * Math.Pow("+Constants.r0+"/r,2);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return V - 2 * V/"+Constants.r0+" * (r - "+Constants.r0+");"); // again linear extrapolation
            stw.WriteLine("         }");
            stw.WriteLine("     }");           
            stw.WriteLine("     public static double VelocityYA(double[] X, double t){");
            stw.WriteLine("         double r = Math.Max(X.L2Norm(), BLAS.MachineEps);");
            stw.WriteLine("         double V = X[1]/r * VelocityA(X,0.0);");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return V * Math.Pow("+Constants.r0+"/r,2);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return V - 2 * V/"+Constants.r0+" * (r - "+Constants.r0+");"); // again linear extrapolation
            stw.WriteLine("         }");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityZA(double[] X, double t){");
            stw.WriteLine("         double r = Math.Max(X.L2Norm(), BLAS.MachineEps);");
            stw.WriteLine("         double V = X[2]/r * VelocityA(X,0.0);");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return V * Math.Pow("+Constants.r0+"/r,2);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return V - 2 * V/"+Constants.r0+" * (r - "+Constants.r0+");"); // again linear extrapolation
            stw.WriteLine("         }");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     static Func<double, double> TempInitial = r => {");
            stw.WriteLine("         return "+Constants.T_wall+"-2*Math.Pow("+Constants.Xi+",2.0)*(("+Constants.rho_B+"*("+Constants.hVap+"+("+Constants.c_A+"-"+Constants.c_B+")*("+Constants.T_wall+"-"+Constants.T_sat+")))/("+Constants.rho_A+"*"+Constants.c_A+"))*MathNet.Numerics.Integration.GaussLegendreRule.Integrate(z => Math.Exp(-Math.Pow("+Constants.Xi+",2.0)*(Math.Pow(1-z,-2.0)-2*"+Constants.eps+"*z-1)),1-"+Constants.r0+"/r,1, 15);");
            stw.WriteLine("     };");
            stw.WriteLine("     public static double TemperatureA(double[] X, double t){");
            stw.WriteLine("         double r = X.L2Norm();");
            stw.WriteLine("         double T_r = "+Constants.Xi+" / ("+Constants.k_A+" / ("+Constants.rho_B+" * "+Constants.hVap+")) * Math.Sqrt("+Constants.alpha_A+" / "+Constants.t0+");");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return TempInitial(r);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return "+Constants.T_sat+" - ("+Constants.r0+"-r) * T_r;");
            stw.WriteLine("         }");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double TemperatureABoundary(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double VelocityB(double[] X, double t){");
            stw.WriteLine("         return 0.0;");
            stw.WriteLine("     }");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }

    static public Formula Get_VAX() {
        return new Formula("BoundaryAndInitialValues.VelocityXA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_VAY() {
        return new Formula("BoundaryAndInitialValues.VelocityYA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_VAZ() {
        return new Formula("BoundaryAndInitialValues.VelocityZA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA() {
        return new Formula("BoundaryAndInitialValues.TemperatureA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA_Boundary() {
        return new Formula("BoundaryAndInitialValues.TemperatureABoundary", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode:GetPrefixCode());
    }
}

In [ ]:
var PhiFuncF = BoundaryAndInitialValueFactory.Get_Phi();
PhiFuncF.Evaluate(new double[3], 0.0).Display();
PhiFuncF.Evaluate(new double[3], Constants.tE).Display();

In [ ]:
Gnuplot plt = new Gnuplot();
plt.PlotFunction(delegate (MultidimensionalArray X, MultidimensionalArray Y) {BoundaryAndInitialValueFactory.Get_TempA().EvaluateV(X,0.0,Y);}, BoSSS.Foundation.Grid.Classic.Grid1D.LineGrid(GenericBlas.Linspace(0,Constants.L,1000)).GridData);
plt.PlotNow()

In [ ]:
Gnuplot plt = new Gnuplot();
plt.PlotFunction(delegate (MultidimensionalArray X, MultidimensionalArray Y) {BoundaryAndInitialValueFactory.Get_VAX().EvaluateV(X,0.0,Y);}, BoSSS.Foundation.Grid.Classic.Grid1D.LineGrid(GenericBlas.Linspace(0,Constants.L,1000)).GridData);
plt.PlotNow()

## Initialize Controlfiles

In [ ]:
int res = 8;
int[] amr = { 0, 1, 2, 3};//{ 0, 1, 2 };
int[] pDeg = { 2 };//{ 2, 3 };
int[] tRef = { 0, 1 };

In [ ]:
List<XNSFE_Control> Controls = new List<XNSFE_Control>();
foreach(int p in pDeg){
    foreach(int lvl in amr){
        foreach(int t in tRef){

            XNSFE_Control C = new XNSFE_Control();

            // basic database options
            // ======================
            C.DbPath      = Path.GetFullPath(BoSSSshell.WorkflowMgm.DefaultDatabase.Path);
            C.SessionName = proj + "_H" + res + "_AMR" + lvl + "_P" + p + "_T" + t ;
            C.ProjectName = proj;
            C.ProjectDescription = "scriven testcase - testcase for evaporation";
            
            C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Degree", p));
            C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("AMR", lvl));
            C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Res", res));
            C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("tRes", t));

            // DG degrees
            // ==========
            C.SetDGdegree(p);
            C.FailOnSolverFail = false;
            // Physical Parameters
            // ===================
            C.PhysicalParameters.Material = false;

            C.PhysicalParameters.IncludeConvection = true;
            C.ThermalParameters.IncludeConvection  = true;

            C.PhysicalParameters.rho_A = Constants.rho_A;
            C.PhysicalParameters.rho_B = Constants.rho_B;

            C.PhysicalParameters.mu_A = Constants.mu_A;
            C.PhysicalParameters.mu_B = Constants.mu_B;

            C.PhysicalParameters.Sigma = Constants.Sigma;

            C.ThermalParameters.rho_A = Constants.rho_A;
            C.ThermalParameters.rho_B = Constants.rho_B;

            C.ThermalParameters.c_A = Constants.c_A;
            C.ThermalParameters.c_B = Constants.c_B;

            C.ThermalParameters.k_A = Constants.k_A;
            C.ThermalParameters.k_B = Constants.k_B;

            C.ThermalParameters.hVap  = Constants.hVap;
            C.ThermalParameters.T_sat = Constants.T_sat;

            C.PhysicalParameters.betaS_A = 0.0;
            C.PhysicalParameters.betaS_B = 0.0;

            C.PhysicalParameters.betaL      = 0.0;
            C.PhysicalParameters.sliplength = 0.0;
            C.PhysicalParameters.theta_e    = Math.PI; 

            // grid generation
            // ===============
            var grd = GridFactory.Grid(res, BoSSSshell.WorkflowMgm.DefaultDatabase);
            C.SetGrid(grd);
            
            // Initial Values
            // ==============
            C.AddInitialValue("Phi", BoundaryAndInitialValueFactory.Get_Phi());
            C.AddInitialValue("Temperature#A", BoundaryAndInitialValueFactory.Get_TempA());
            C.AddInitialValue("Temperature#B", "X => "+Constants.T_sat+"", false);
            C.AddInitialValue("VelocityX#A", BoundaryAndInitialValueFactory.Get_VAX());
            C.AddInitialValue("VelocityY#A", BoundaryAndInitialValueFactory.Get_VAY());
            C.AddInitialValue("VelocityZ#A", BoundaryAndInitialValueFactory.Get_VAZ());

            // boundary conditions
            // ===================
            C.AddBoundaryValue("freeslip_ZeroGradient");
            C.AddBoundaryValue("pressure_outlet_ConstantTemperature", "Temperature#A", BoundaryAndInitialValueFactory.Get_TempA_Boundary());

            // level set options
            // =================
            C.Option_LevelSetEvolution                        = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
            C.Timestepper_LevelSetHandling                    = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;
            C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
            C.ReInitPeriod = 10;

            // Timestepping
            // ============
            C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
            C.dtFixed          = Math.Pow(0.5, t) * Constants.dt((int)(res * Math.Pow(2, amr.Max())), pDeg.Max()); // Use same timestep for all, better comparability
            C.Endtime          = Constants.tE;
            C.NoOfTimesteps    = (int)Math.Ceiling(C.Endtime /C.dtFixed);
            C.saveperiod       = 1;
            C.rollingSaves     = false;
            
            // AMR
            // ============
            int level = lvl;
            C.AdaptiveMeshRefinement = lvl > 0;
            C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ maxRefinementLevel = level, bandwidth=0});
            C.AMR_startUpSweeps = lvl;

            // grid partition
            // ============
            // C.GridPartType = GridPartType.Predefined;
            // C.DynamicLoadBalancing_On = false;
            // C.DynamicLoadBalancing_RedistributeAtStartup = true;
            C.GridPartType = GridPartType.METIS;
            C.DynamicLoadBalancing_On = true;
            C.DynamicLoadBalancing_RedistributeAtStartup = true;
            C.DynamicLoadBalancing_Period = 10;
            C.DynamicLoadBalancing_CellCostEstimators.Clear();
            C.DynamicLoadBalancing_CellCostEstimators.Add(new BoSSS.Application.XNSE_Solver.Loadbalancing.XNSECellCostEstimator() );

            // special options
            // ===============
            //C.LinearSolver = LinearSolverCode.direct_mumps.GetConfig(); // seemed more stable than pardiso
            C.NonLinearSolver.MaxSolverIterations = 20;
            //C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.Dropletlike() { LogPeriod = 1 }); // give the outlet edgetag

            Controls.Add(C);
        }
    }
}

In [ ]:
int ctrlCount = Controls.Count();
ctrlCount

In [ ]:
int numproc = 4;
var jobs = Controls.Select(c => {c.SessionName = c.SessionName + "_MPI" + numproc; return c.CreateJob();}).ToArray();
jobs.ForEach(j => j.NumberOfMPIProcs = numproc);
jobs.ForEach(j => j.NumberOfThreads = 1);
jobs.ForEach(j => j.EnvironmentVars["BOSSS_ARG_7"] = j.NumberOfThreads.ToString());
jobs.Activate();

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(172800);

In [ ]:
int count = BoSSSshell.wmg.Sessions.Count();
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(count != ctrlCount || count != success){
    throw new ApplicationException("Not all simulations calculated or finished successful!");
}

## Begin Postprocessing

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination);
sessions = sessions.OrderBy(s => s.KeysAndQueries["id:AMR"]).ToList();
sessions

In [ ]:
sessions = sessions.Pick(0, 2, 3);

Lets sort the sessions by degree

In [ ]:
Dictionary<int, List<ISessionInfo>> groupedSessions = sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList());

Load Log Data

In [ ]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));

### Growthrate


### Interface Position

Evaluate the analyical solution for the interface position

In [ ]:
double[] times = GenericBlas.Linspace(0.0, Constants.tE, 20);
double[] yValues = new double[times.Length];
var PhiFuncF = BoundaryAndInitialValueFactory.Get_Phi();
Func<double, double> PhiFunc = t => {
    return PhiFuncF.Evaluate(new double[3], t);
};
for(int i = 0; i < times.Length; i++){
    yValues[i] = PhiFunc(times[i]);
}
Plot2Ddata RefPos = new Plot2Ddata(times, yValues, "analytical Interface Position");

In [ ]:
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    var plot = dat.ElementAt(0).Merge(RefPos);
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"i", "t", "l"}; 
    var p1 = plot.PlotNow(); 
    
    plot.ShowLegend = false;
    plot.XrangeMin = Constants.tE - Constants.t0;
    plot.XrangeMax = Constants.tE;

    plot.YrangeMin = 0.99 * PhiFunc(plot.XrangeMin.Value);
    plot.YrangeMax = 1.01 * PhiFunc(plot.XrangeMax.Value);
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

Repeat the same for positional errors

In [ ]:
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    Plot2Ddata PositionErrorAbs = new Plot2Ddata();
    Plot2Ddata PositionErrorRel = new Plot2Ddata();
    foreach(var dataGroup in dat.ElementAt(0).dataGroups){
        double[] times = dataGroup.Abscissas;
        double[] yValuesAbs = new double[times.Length];
        double[] yValuesRel = new double[times.Length];
        for(int i = 0; i < times.Length; i++){
            double y = dataGroup.Values[i];
            double y_ex = PhiFunc(times[i]);
            yValuesAbs[i] = y - y_ex;
            yValuesRel[i] = (y - y_ex)/y_ex;

        }
        Plot2Ddata errDataAbs = new Plot2Ddata(times, yValuesAbs, "abs-err-" + dataGroup.Name);
        PositionErrorAbs      = PositionErrorAbs.Merge(errDataAbs);
        Plot2Ddata errDataRel = new Plot2Ddata(times, yValuesRel, "rel-err-" + dataGroup.Name);
        PositionErrorRel      = PositionErrorRel.Merge(errDataRel);

    }

    // absolute errors
    var plot = PositionErrorAbs;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p1 = plot.PlotNow(); 
    
    plot = PositionErrorRel;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

### Animation of Temperature and Velocity - INCOMPLETE

In [ ]:
var sess2plot = sessions.First();//groupedSessions[groupedSessions.Keys.Max()].OrderBy(s => s.KeysAndQueries["Grid:hMax"]).First();// pick session with finest grid and highest degree
string path2plot = Path.GetFullPath(Path.Combine("./Plot", sess2plot.Name));
sess2plot.Export().To(path2plot).WithSupersampling(2).WithShadowFields().Do(true);

In [ ]:
#!pwsh
#!share --from csharp path2plot
pvpython ./paraviewscript/scriventestcase.py $path2plot

In [ ]:
display(new HtmlString(String.Format("<video width=\"1463\" height=\"761\" controls><source src=\"{0}\" type=\'video/ogg; codecs=\"theora\"\'></video>",Path.Combine("./Plot", sess2plot.Name, "scriventestcase.ogv") )))

Add more postprocessing, look up what you have already done last year Matthias!  
Especially how to obtain some meaningful EOC?

$\int_{\Omega} c dV$

#### Alternative Postprocessing, loading timesteps, and estimating radius depending on volume

In [ ]:
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Quadrature;

In [ ]:
public static Plot2Ddata GetRadius(IEnumerable<ISessionInfo> s){
    Plot2Ddata plt = new Plot2Ddata();
    foreach(var _s in s){
        plt = plt.Merge(GetRadius(_s));
    }
    return plt;
}

public static Plot2Ddata GetRadius(ISessionInfo s){
    List<ISessionInfo> sess = new List<ISessionInfo>();
    sess.Add(s);
    loadsessions(s);

    void loadsessions(ISessionInfo _s){
        if(_s.RestartedFrom != Guid.Empty){
            Console.WriteLine("{0} restarted from {1}, loading as well ...", _s.ID, _s.RestartedFrom);
            ISessionInfo restart = _s.Database.Controller.DBDriver.LoadSession(_s.RestartedFrom, _s.Database);
            sess.Add(restart);
            loadsessions(restart);
        }
    }
    var ts = sess.SelectMany(_s => _s.Timesteps.Where(t => t.TimeStepNumber.Length == 1)); 
    ts = ts.OrderBy(t => t.PhysicalTime);
    Console.WriteLine("{0} has {1} timesteps in total", s.ID, ts.Count());

    double[] times = new double[ts.Count()];
    double[] yValues = new double[times.Length];
    for(int i = 0; i < times.Length; i++){
        var _ts = ts.ElementAt(i);
        times[i] = _ts.PhysicalTime;

        var phi = (LevelSet)_ts.Fields.SingleOrDefault(f => f.Identification == "Phi");
        var trk = new LevelSetTracker((GridData)phi.GridDat, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new[] {"A", "B"}, phi);
        trk.UpdateTracker(0.0);
        var vol = BoSSS.Solution.XNSECommon.XNSEUtils.GetSpeciesArea(trk, trk.GetSpeciesId("B"));
        // vol = 4/3 * pi * r^3 * 1/8
        // r = (6 * vol / pi)^1/3
        double r = Math.Pow(6 * vol / Math.PI, 1.0/3.0);
        yValues[i] = r;
    }
    Plot2Ddata Pos = new Plot2Ddata(times, yValues, s.Name);
    return Pos;
}

In [ ]:
sessions

In [ ]:
Constants.tE

In [ ]:
Convert.ToDouble(sessions.First().KeysAndQueries["dtFixed"]) * (Convert.ToDouble(sessions.First().KeysAndQueries["NoOfTimesteps"]))

In [ ]:
sessions.Pick(2).GetApproximateRunTime()

In [ ]:
var plt = GetRadius(sessions);

In [ ]:
plt = plt.Merge(RefPos);
plt.ModFormat();


In [ ]:
plt.PlotNow()